In [ ]:
%pip install pyspark

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EV Charging ETL Pipeline").getOrCreate()

In [6]:
file_path = "../data/raw/acndata_sessions.json"


In [7]:
raw_df = spark.read.option("multiLine", True).json(file_path)

raw_df.printSchema()

root
 |-- _items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- _id: string (nullable = true)
 |    |    |-- clusterID: string (nullable = true)
 |    |    |-- connectionTime: string (nullable = true)
 |    |    |-- disconnectTime: string (nullable = true)
 |    |    |-- doneChargingTime: string (nullable = true)
 |    |    |-- kWhDelivered: double (nullable = true)
 |    |    |-- sessionID: string (nullable = true)
 |    |    |-- siteID: string (nullable = true)
 |    |    |-- spaceID: string (nullable = true)
 |    |    |-- stationID: string (nullable = true)
 |    |    |-- timezone: string (nullable = true)
 |    |    |-- userID: string (nullable = true)
 |    |    |-- userInputs: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- WhPerMile: long (nullable = true)
 |    |    |    |    |-- kWhRequested: double (nullable = true)
 |    |    |    |    |-- milesRequested: long (nullable = tru

In [17]:
from pyspark.sql.functions import explode

sessions_df = raw_df.select(
    explode("_items").alias("session")
)

sessions_df = sessions_df.select("session.*")

In [ ]:
sessions_df.printSchema()

print("Number of sessions:", sessions_df.count())

sessions_df.show(5, truncate=False)

root
 |-- _id: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: string (nullable = true)
 |-- disconnectTime: string (nullable = true)
 |-- doneChargingTime: string (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |    |-- paymentRequired: boolean (nullable = true)
 |    |    |-- requestedDeparture: string (nullable = true)
 |    |    |-- userID: lon

In [22]:
from pyspark.sql.functions import col, sum

In [23]:
missing_values = sessions_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in sessions_df.columns
    ]
    
)

missing_values.show()

+---+---------+--------------+--------------+----------------+------------+---------+------+-------+---------+--------+------+----------+
|_id|clusterID|connectionTime|disconnectTime|doneChargingTime|kWhDelivered|sessionID|siteID|spaceID|stationID|timezone|userID|userInputs|
+---+---------+--------------+--------------+----------------+------------+---------+------+-------+---------+--------+------+----------+
|  0|        0|             0|             0|               1|           0|        0|     0|      0|        0|       0|  2483|      2483|
+---+---------+--------------+--------------+----------------+------------+---------+------+-------+---------+--------+------+----------+



In [24]:
sessions_df.filter(
    col("doneChargingTime").isNull()
).show(truncate=False)

+------------------------+---------+-----------------------------+-----------------------------+----------------+------------+--------------------------------------+------+-------+-----------+-------------------+------+----------+
|_id                     |clusterID|connectionTime               |disconnectTime               |doneChargingTime|kWhDelivered|sessionID                             |siteID|spaceID|stationID  |timezone           |userID|userInputs|
+------------------------+---------+-----------------------------+-----------------------------+----------------+------------+--------------------------------------+------+-------+-----------+-------------------+------+----------+
|5bc915caf9af8b0dad3c0665|0039     |Mon, 30 Apr 2018 15:27:36 GMT|Mon, 30 Apr 2018 19:31:44 GMT|NULL            |11.676      |2_39_138_29_2018-04-30 15:27:36.253326|0002  |CA-304 |2-39-138-29|America/Los_Angeles|NULL  |NULL      |
+------------------------+---------+-----------------------------+----------

In [25]:
duplicate_sessions = sessions_df.groupBy("sessionID").count().filter(col("count") > 1)

duplicate_sessions.show()

+---------+-----+
|sessionID|count|
+---------+-----+
+---------+-----+



In [29]:
from pyspark.sql.functions import col, regexp_replace, to_timestamp

sessions_df = sessions_df.withColumn(
    "connectionTime",
    to_timestamp(
        regexp_replace(col("connectionTime"), r"^[A-Za-z]{3}, | GMT$", ""),
        "dd MMM yyyy HH:mm:ss"
    )
).withColumn(
    "disconnectTime",
    to_timestamp(
        regexp_replace(col("disconnectTime"), r"^[A-Za-z]{3}, | GMT$", ""),
        "dd MMM yyyy HH:mm:ss"
    )
).withColumn(
    "doneChargingTime",
    to_timestamp(
        regexp_replace(col("doneChargingTime"), r"^[A-Za-z]{3}, | GMT$", ""),
        "dd MMM yyyy HH:mm:ss"
    )
)

In [31]:
sessions_df.printSchema()
sessions_df.select(
    "connectionTime",
    "disconnectTime",
    "doneChargingTime"
).show(5, truncate=False)

root
 |-- _id: string (nullable = true)
 |-- clusterID: string (nullable = true)
 |-- connectionTime: timestamp (nullable = true)
 |-- disconnectTime: timestamp (nullable = true)
 |-- doneChargingTime: timestamp (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- sessionID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- spaceID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- userID: string (nullable = true)
 |-- userInputs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- WhPerMile: long (nullable = true)
 |    |    |-- kWhRequested: double (nullable = true)
 |    |    |-- milesRequested: long (nullable = true)
 |    |    |-- minutesAvailable: long (nullable = true)
 |    |    |-- modifiedAt: string (nullable = true)
 |    |    |-- paymentRequired: boolean (nullable = true)
 |    |    |-- requestedDeparture: string (nullable = true)
 |    |    |-- us

In [32]:
invalid_sessions = sessions_df.filter(
    col("disconnectTime") < col("connectionTime")
)

print("Invalid sessions:", invalid_sessions.count())

Invalid sessions: 0


In [33]:
from pyspark.sql.functions import unix_timestamp

In [34]:
sessions_df = sessions_df.withColumn(
    "charging_duration_minutes",
    (
        unix_timestamp("doneChargingTime") -
        unix_timestamp("connectionTime")
    ) / 60
).withColumn(
    "connection_duration_minutes",
    (
        unix_timestamp("disconnectTime") -
        unix_timestamp("connectionTime")
    ) / 60
)

In [35]:
sessions_df.select(
    "connectionTime",
    "doneChargingTime",
    "disconnectTime",
    "charging_duration_minutes",
    "connection_duration_minutes"
).show(10, truncate=False)

+-------------------+-------------------+-------------------+-------------------------+---------------------------+
|connectionTime     |doneChargingTime   |disconnectTime     |charging_duration_minutes|connection_duration_minutes|
+-------------------+-------------------+-------------------+-------------------------+---------------------------+
|2018-04-25 11:08:04|2018-04-25 13:21:10|2018-04-25 13:20:10|133.1                    |132.1                      |
|2018-04-25 13:45:10|2018-04-25 16:44:15|2018-04-26 00:56:16|179.08333333333334       |671.1                      |
|2018-04-25 13:45:50|2018-04-25 14:51:44|2018-04-25 23:04:45|65.9                     |558.9166666666666          |
|2018-04-25 14:37:06|2018-04-25 16:05:22|2018-04-25 23:55:34|88.26666666666667        |558.4666666666667          |
|2018-04-25 14:40:34|2018-04-25 17:40:30|2018-04-25 23:03:12|179.93333333333334       |502.6333333333333          |
|2018-04-25 14:43:50|2018-04-25 16:18:28|2018-04-26 01:17:30|94.63333333

In [37]:
sessions_df.filter(
    col("charging_duration_minutes") < 0
).select(
    "sessionID",
    "connectionTime",
    "doneChargingTime",
    "disconnectTime",
    "kWhDelivered",
    "stationID"
).show(truncate=False)

+---------------------------------------+-------------------+-------------------+-------------------+------------------+------------+
|sessionID                              |connectionTime     |doneChargingTime   |disconnectTime     |kWhDelivered      |stationID   |
+---------------------------------------+-------------------+-------------------+-------------------+------------------+------------+
|2_39_78_363_2018-05-04 19:08:36.642114 |2018-05-04 19:08:37|2018-05-04 19:07:40|2018-05-04 22:07:47|0.5517219235364658|2-39-78-363 |
|2_39_78_367_2018-05-04 19:23:51.897392 |2018-05-04 19:23:52|2018-05-04 19:22:52|2018-05-05 00:04:15|0.9122968936674426|2-39-78-367 |
|2_39_139_567_2018-05-07 20:47:50.862655|2018-05-07 20:47:51|2018-05-07 20:47:50|2018-05-08 02:16:00|14.967            |2-39-139-567|
+---------------------------------------+-------------------+-------------------+-------------------+------------------+------------+



In [38]:
from pyspark.sql.functions import when

sessions_df = sessions_df.withColumn(
    "charging_duration_minutes",
    when(
        col("doneChargingTime") >= col("connectionTime"),
        (
            unix_timestamp("doneChargingTime") -
            unix_timestamp("connectionTime")
        ) / 60
    )
)

In [39]:
sessions_df.select(
    "sessionID",
    "connectionTime",
    "doneChargingTime",
    "charging_duration_minutes"
).filter(
    col("charging_duration_minutes").isNull()
).show(truncate=False)

+---------------------------------------+-------------------+-------------------+-------------------------+
|sessionID                              |connectionTime     |doneChargingTime   |charging_duration_minutes|
+---------------------------------------+-------------------+-------------------+-------------------------+
|2_39_138_29_2018-04-30 15:27:36.253326 |2018-04-30 15:27:36|NULL               |NULL                     |
|2_39_78_363_2018-05-04 19:08:36.642114 |2018-05-04 19:08:37|2018-05-04 19:07:40|NULL                     |
|2_39_78_367_2018-05-04 19:23:51.897392 |2018-05-04 19:23:52|2018-05-04 19:22:52|NULL                     |
|2_39_139_567_2018-05-07 20:47:50.862655|2018-05-07 20:47:51|2018-05-07 20:47:50|NULL                     |
+---------------------------------------+-------------------+-------------------+-------------------------+



In [40]:
sessions_df.select(
    "charging_duration_minutes",
    "connection_duration_minutes"
).describe().show()

+-------+-------------------------+---------------------------+
|summary|charging_duration_minutes|connection_duration_minutes|
+-------+-------------------------+---------------------------+
|  count|                     2495|                       2499|
|   mean|       212.95420841683324|         379.42607709750604|
| stddev|       192.06524615423308|         360.27042420349824|
|    min|                      0.0|          5.733333333333333|
|    max|       1824.2666666666667|          5075.666666666667|
+-------+-------------------------+---------------------------+



In [41]:
from pyspark.sql.functions import (
    to_date,
    hour,
    dayofweek,
    month
)

In [42]:
sessions_df = sessions_df.withColumn(
    "charging_date",
    to_date("connectionTime")
).withColumn(
    "hour",
    hour("connectionTime")
).withColumn(
    "day_of_week",
    dayofweek("connectionTime")
).withColumn(
    "month",
    month("connectionTime")
)

In [43]:
sessions_df.select(
    "connectionTime",
    "charging_date",
    "hour",
    "day_of_week",
    "month"
).show(10, truncate=False)

+-------------------+-------------+----+-----------+-----+
|connectionTime     |charging_date|hour|day_of_week|month|
+-------------------+-------------+----+-----------+-----+
|2018-04-25 11:08:04|2018-04-25   |11  |4          |4    |
|2018-04-25 13:45:10|2018-04-25   |13  |4          |4    |
|2018-04-25 13:45:50|2018-04-25   |13  |4          |4    |
|2018-04-25 14:37:06|2018-04-25   |14  |4          |4    |
|2018-04-25 14:40:34|2018-04-25   |14  |4          |4    |
|2018-04-25 14:43:50|2018-04-25   |14  |4          |4    |
|2018-04-25 14:47:42|2018-04-25   |14  |4          |4    |
|2018-04-25 14:58:25|2018-04-25   |14  |4          |4    |
|2018-04-25 15:10:52|2018-04-25   |15  |4          |4    |
|2018-04-25 15:12:11|2018-04-25   |15  |4          |4    |
+-------------------+-------------+----+-----------+-----+
only showing top 10 rows


In [44]:
sessions_df.select(
    min("connectionTime").alias("earliest_session"),
    max("connectionTime").alias("latest_session")
).show()

+-------------------+-------------------+
|   earliest_session|     latest_session|
+-------------------+-------------------+
|2018-04-25 11:08:04|2018-06-08 15:29:16|
+-------------------+-------------------+



In [45]:
print("Unique sites:", sessions_df.select("siteID").distinct().count())
print("Unique stations:", sessions_df.select("stationID").distinct().count())
print("Unique charging spaces:", sessions_df.select("spaceID").distinct().count())

Unique sites: 1
Unique stations: 52
Unique charging spaces: 52


In [46]:
#Which hours of the day have the most charging sessions?

hourly_demand = sessions_df.groupBy("hour").count() \
    .withColumnRenamed("count", "session_count") \
    .orderBy("hour")

hourly_demand.show(24)


+----+-------------+
|hour|session_count|
+----+-------------+
|   0|          121|
|   1|          151|
|   2|           99|
|   3|           87|
|   4|           58|
|   5|           41|
|   6|           21|
|   7|            9|
|   8|            9|
|   9|            3|
|  10|            2|
|  11|           39|
|  12|           15|
|  13|           70|
|  14|          174|
|  15|          519|
|  16|          331|
|  17|          173|
|  18|           87|
|  19|          111|
|  20|          130|
|  21|           70|
|  22|           81|
|  23|           98|
+----+-------------+



In [51]:
from pyspark.sql.functions import sum, count, avg

hourly_demand = sessions_df.groupBy("hour").agg(
    count("sessionID").alias("session_count"),
    sum("kWhDelivered").alias("total_kWh"),
    avg("kWhDelivered").alias("avg_kWh_per_session")
).orderBy("hour")

hourly_demand.show(24)

+----+-------------+------------------+-------------------+
|hour|session_count|         total_kWh|avg_kWh_per_session|
+----+-------------+------------------+-------------------+
|   0|          121|1281.9946833930703| 10.594997383413805|
|   1|          151|          1343.862|  8.899748344370861|
|   2|           99|           997.553|  10.07629292929293|
|   3|           87| 771.7259999999999|  8.870413793103447|
|   4|           58| 707.8173188888888| 12.203746877394636|
|   5|           41|422.53600000000006| 10.305756097560977|
|   6|           21|           203.672|  9.698666666666666|
|   7|            9|            77.866|  8.651777777777777|
|   8|            9|           103.524| 11.502666666666666|
|   9|            3|            60.184| 20.061333333333334|
|  10|            2|17.256999999999998|  8.628499999999999|
|  11|           39| 409.5379227628425| 10.500972378534424|
|  12|           15| 99.77199999999998|  6.651466666666665|
|  13|           70|            620.43| 

In [48]:
#Which charging stations are busiest?

station_summary = sessions_df.groupBy("stationID").agg(
    count("sessionID").alias("session_count"),
    sum("kWhDelivered").alias("total_kWh"),
    avg("kWhDelivered").alias("avg_kWh_per_session"),
    avg("connection_duration_minutes").alias("avg_connection_minutes")
).orderBy(
    col("session_count").desc()
)

station_summary.show(10, truncate=False)

+-----------+-------------+-----------------+-------------------+----------------------+
|stationID  |session_count|total_kWh        |avg_kWh_per_session|avg_connection_minutes|
+-----------+-------------+-----------------+-------------------+----------------------+
|2-39-139-28|107          |1031.297         |9.638289719626169  |336.6559190031154     |
|2-39-138-29|90           |663.8399999999997|7.375999999999997  |225.53629629629629    |
|2-39-129-17|85           |688.99           |8.105764705882352  |285.0594117647059     |
|2-39-127-19|85           |656.0100000000001|7.717764705882354  |264.26333333333326    |
|2-39-131-30|79           |730.073          |9.241430379746836  |493.75907172995784    |
|2-39-79-380|76           |657.6819999999998|8.653710526315786  |348.95767543859654    |
|2-39-123-23|76           |627.6429999999998|8.258460526315787  |314.8203947368422     |
|2-39-79-377|73           |711.1259999999996|9.741452054794516  |420.9785388127854     |
|2-39-79-379|72      

In [52]:
station_summary.orderBy(
    col("total_kWh").desc()
).show(5, truncate=False)

+-----------+-------------+-----------------+-------------------+----------------------+
|stationID  |session_count|total_kWh        |avg_kWh_per_session|avg_connection_minutes|
+-----------+-------------+-----------------+-------------------+----------------------+
|2-39-139-28|107          |1031.297         |9.638289719626169  |336.6559190031154     |
|2-39-91-437|64           |883.1283661111111|13.79888072048611  |460.9203125000001     |
|2-39-95-27 |59           |739.9839999999999|12.542101694915253 |525.9672316384181     |
|2-39-131-30|79           |730.073          |9.241430379746836  |493.75907172995784    |
|2-39-79-379|72           |714.377          |9.921902777777778  |567.0770833333333     |
+-----------+-------------+-----------------+-------------------+----------------------+
only showing top 5 rows


In [53]:
#How long do vehicles typically stay connected, and how much energy do they receive?

sessions_df.select(
    "kWhDelivered",
    "charging_duration_minutes",
    "connection_duration_minutes"
).describe().show()

+-------+-----------------+-------------------------+---------------------------+
|summary|     kWhDelivered|charging_duration_minutes|connection_duration_minutes|
+-------+-----------------+-------------------------+---------------------------+
|  count|             2499|                     2495|                       2499|
|   mean|8.899330884942955|       212.95420841683324|         379.42607709750604|
| stddev|6.623244093713143|       192.06524615423308|         360.27042420349824|
|    min|            0.501|                      0.0|          5.733333333333333|
|    max|           47.808|       1824.2666666666667|          5075.666666666667|
+-------+-----------------+-------------------------+---------------------------+



In [54]:
#creating final datasets

charging_sessions = sessions_df.select(
    "sessionID",
    "stationID",
    "siteID",
    "connectionTime",
    "doneChargingTime",
    "disconnectTime",
    "kWhDelivered",
    "charging_duration_minutes",
    "connection_duration_minutes",
    "charging_date",
    "hour",
    "day_of_week",
    "month"
)

In [57]:
charging_sessions.printSchema()
charging_sessions.show(5, truncate=False)
print("Rows:", charging_sessions.count())
print("Columns:", len(charging_sessions.columns))

root
 |-- sessionID: string (nullable = true)
 |-- stationID: string (nullable = true)
 |-- siteID: string (nullable = true)
 |-- connectionTime: timestamp (nullable = true)
 |-- doneChargingTime: timestamp (nullable = true)
 |-- disconnectTime: timestamp (nullable = true)
 |-- kWhDelivered: double (nullable = true)
 |-- charging_duration_minutes: double (nullable = true)
 |-- connection_duration_minutes: double (nullable = true)
 |-- charging_date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- month: integer (nullable = true)

+--------------------------------------+-----------+------+-------------------+-------------------+-------------------+------------+-------------------------+---------------------------+-------------+----+-----------+-----+
|sessionID                             |stationID  |siteID|connectionTime     |doneChargingTime   |disconnectTime     |kWhDelivered|charging_duration_minutes|connection_duration_m

In [58]:
station_summary = station_summary.select(
    "stationID",
    "session_count",
    "total_kWh",
    "avg_kWh_per_session",
    "avg_connection_minutes"
)

In [59]:
station_summary.show(10, truncate=False)

print("Stations:", station_summary.count())
print("Columns:", len(station_summary.columns))

+-----------+-------------+-----------------+-------------------+----------------------+
|stationID  |session_count|total_kWh        |avg_kWh_per_session|avg_connection_minutes|
+-----------+-------------+-----------------+-------------------+----------------------+
|2-39-139-28|107          |1031.297         |9.638289719626169  |336.6559190031154     |
|2-39-138-29|90           |663.8399999999997|7.375999999999997  |225.53629629629629    |
|2-39-129-17|85           |688.99           |8.105764705882352  |285.0594117647059     |
|2-39-127-19|85           |656.0100000000001|7.717764705882354  |264.26333333333326    |
|2-39-131-30|79           |730.073          |9.241430379746836  |493.75907172995784    |
|2-39-79-380|76           |657.6819999999998|8.653710526315786  |348.95767543859654    |
|2-39-123-23|76           |627.6429999999998|8.258460526315787  |314.8203947368422     |
|2-39-79-377|73           |711.1259999999996|9.741452054794516  |420.9785388127854     |
|2-39-79-379|72      

In [8]:
processed_path = "../data/processed"

In [61]:
charging_sessions.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(processed_path + "/charging_sessions")

In [62]:
station_summary.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(processed_path + "/station_summary")